In [2]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
!ls data

2016-17-all-secondary-stats-data.xlsx
2017-18-all-secondary-stats-data.xlsx
2018-19-all-secondary-stats-data.xlsx
2019-20-all-secondary-stats-data.xlsx
2020-21-all-secondary-stats-data.xlsx
2022-23-all-secondary-stats-data-completed.xlsx
2023-24-all-secondary-stats-data-complete-1.xlsx
2024-25-all-secondary-stats-data.xlsx


In [4]:
import glob
import os

# Define the path to your data directory
path = 'data/'

# Use glob to find all files matching the pattern
# This is flexible and will find any file with "secondary-stats" in its name
all_files = glob.glob(os.path.join(path, "*-secondary-stats*.xlsx"))

# Create an empty dictionary to store the DataFrames
dataframes = {}

print("Found files to load:")
for filename in all_files:
    # Extract a clean name from the filename to use as a dictionary key
    # e.g., 'data/2016-17-all-secondary-stats-data.xlsx' becomes '2016-17'
    key = os.path.basename(filename).split('-all-')[0]
    
    # Load the Excel file into a DataFrame and add it to the dictionary
    dataframes[key] = pd.read_excel(filename)
    print(f"- {os.path.basename(filename)} (loaded as dataframes['{key}'])")

# Now you can access each individual DataFrame
print("\n--- Example: Accessing the first 5 rows of the 2018-19 data ---")
print(dataframes['2018-19'].head())

Found files to load:
- 2024-25-all-secondary-stats-data.xlsx (loaded as dataframes['2024-25'])
- 2019-20-all-secondary-stats-data.xlsx (loaded as dataframes['2019-20'])
- 2018-19-all-secondary-stats-data.xlsx (loaded as dataframes['2018-19'])
- 2017-18-all-secondary-stats-data.xlsx (loaded as dataframes['2017-18'])
- 2022-23-all-secondary-stats-data-completed.xlsx (loaded as dataframes['2022-23'])
- 2023-24-all-secondary-stats-data-complete-1.xlsx (loaded as dataframes['2023-24'])
- 2020-21-all-secondary-stats-data.xlsx (loaded as dataframes['2020-21'])
- 2016-17-all-secondary-stats-data.xlsx (loaded as dataframes['2016-17'])

--- Example: Accessing the first 5 rows of the 2018-19 data ---
                                         School Name  Number on roll Jan 2018  \
0             *Harrogate High School (Academy 11-18)                    555.0   
1            *Outwood Academy Ripon  (Academy 11-18)                    656.0   
2  *St John Fisher Catholic High School (Voluntar...      

In [20]:
import pandas as pd
import glob
import os
import re

# 1. Load all files into a dictionary (this part is unchanged)
path = 'data/'
all_files = glob.glob(os.path.join(path, "*-secondary-stats*.xlsx"))
dataframes = {}
for filename in all_files:
    key = os.path.basename(filename).split('-all-')[0]
    dataframes[key] = pd.read_excel(filename)

# 2. Define the mapping for all NON-CONFLICTING columns
column_mapping = {
    'School Name': 'School Name',
    'Type of Establishment': 'Type of Establishment',
    'Number on roll': r'Number on roll Jan',
    'Total applications received': r'Total number of applications received',
    'Places allocated': r'Total Number of places allocated|Total number of pupils allocated',
    'Last child criteria': r'Criteria of the last child allocated',
    'Distance of last child allocated': r'\*Distance .*of last child allocated',
    'Waiting list size': r'\*Waiting list as at',
    'Appeals heard': r'\*Number of appeals heard',
    'Appeals upheld': r'\*Number of appeals upheld'
}

# 3. Process each DataFrame with the corrected logic
processed_dfs = []
for year, df in dataframes.items():
    df_copy = df.copy()
    rename_dict = {}
    
    for old_col in df_copy.columns:
        col_str = str(old_col).lower()

        if 'pan for academic year' in col_str and 'sixth form' in col_str:
            rename_dict[old_col] = 'PAN (sixth form)'
        elif 'pan for academic year' in col_str:
            rename_dict[old_col] = 'PAN'
        else:
            for new_col, pattern in column_mapping.items():
                if re.search(pattern, str(old_col), re.IGNORECASE):
                    rename_dict[old_col] = new_col
                    break
    
    df_copy.rename(columns=rename_dict, inplace=True)
    df_copy['Year'] = year
    
    # 💡 THE FIX IS HERE:
    # We must use .keys() to get the list of final, clean column names.
    final_cols_to_keep = ['School Name', 'Type of Establishment', 'PAN', 'PAN (sixth form)', 'Year'] + list(column_mapping.keys())
    
    # Filter the DataFrame to only include the columns we successfully renamed and want to keep.
    existing_cols_in_df = [col for col in final_cols_to_keep if col in df_copy.columns]
    
    processed_dfs.append(df_copy[existing_cols_in_df])

# 4. Concatenate all the processed DataFrames
final_df = pd.concat(processed_dfs, ignore_index=True)

print("--- Final Combined DataFrame ---")
print(final_df.head())

print("\n--- Final Columns ---")
print(final_df.columns.tolist())

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [34]:
processed_dfs[0]['School Name']

,School Name,School Name
0,Barlby High School,Barlby High School
1,Bedale High School,Bedale High School
2,Boroughbridge High School,Boroughbridge High School
3,Brayton Academy,Brayton Academy
4,Caedmon College Whitby,Caedmon College Whitby
5,Ermysted’s Grammar School for Boys,Ermysted’s Grammar School for Boys
6,Filey School,Filey School
7,George Pindar School,George Pindar School
8,Graham School,Graham School
9,Harrogate Grammar School,Harrogate Grammar School


In [35]:
import pandas as pd

# Create a new list to hold the DataFrames after cleaning
cleaned_dfs = []
for df in processed_dfs:
    # The fix: Select all rows, but only the columns that are not duplicates.
    # The ~ inverts the boolean mask, keeping the first occurrence of each column name.
    cleaned_df = df.loc[:, ~df.columns.duplicated()]
    cleaned_dfs.append(cleaned_df)

# Now, run your filtering on the cleaned list of DataFrames
filtered_list = [
    df[df['School Name'].str.contains('Ripon Grammar', case=False, na=False)] 
    for df in cleaned_dfs  # <-- Use the new 'cleaned_dfs' list here
]

# Combine the data from the filtered list into a single DataFrame
ripon_df = pd.concat(filtered_list, ignore_index=True)

print("--- Data for schools matching 'Ripon Grammar' (after removing duplicates) ---")
print(ripon_df)

--- Data for schools matching 'Ripon Grammar' (after removing duplicates) ---
                               School Name Type of Establishment  \
0                     Ripon Grammar School      Community School   
1  Ripon Grammar School (Community 11 -18)                   NaN   
2  Ripon Grammar School (Community 11 -18)                   NaN   
3  Ripon Grammar School (Community 11 -18)                   NaN   
4                     Ripon Grammar School      Community school   
5                     Ripon Grammar School      Community school   
6  Ripon Grammar School (Community 11 -18)                   NaN   
7  Ripon Grammar School (Community 11 -18)                   NaN   

                                               PAN PAN (sixth form)     Year  \
0                       117 inc 14 Boarding places             40.0  2024-25   
1                      117 (including 14 boarders)               30  2019-20   
2  117 (including 14 boarders) (sixth form PAN 30)              NaN  

In [41]:
ripon_df.sort_values(by='Year')[['Year', 'Distance of last child allocated']].values

array([['2016-17', '13.74 (Day) & 26.66 (Boarding)'],
       ['2017-18', '12.48 (Day) & 4,136 (Boarding)'],
       ['2018-19', '6.68 (Day) & 942 (Boarding)'],
       ['2019-20', '6.68 (Day) & 9.42 (Boarding)'],
       ['2020-21', '8.99 miles (Day) & 52.41  miles (Boarding)'],
       ['2022-23',
        '20.36 miles (Day) 120.86 miles (termly boarding place) 889 miles (weekly boarding place)'],
       ['2023-24',
        '11.07 miles (Day) 82.60 miles (termly boarding place) 63.60 miles (weekly boarding place)'],
       ['2024-25', '7.13 miles (Day)  ']], dtype=object)

In [36]:
filtered_list

[             School Name Type of Establishment                         PAN  \
 24  Ripon Grammar School      Community School  117 inc 14 Boarding places   
 
     PAN (sixth form)     Year  Number on roll  Total applications received  \
 24              40.0  2024-25           926.0                        304.0   
 
     Places allocated     Last child criteria Distance of last child allocated  \
 24             117.0  Out Catchment Distance               7.13 miles (Day)     
 
    Waiting list size Appeals heard Appeals upheld  
 24                26            14              2  ,
                                School Name                          PAN  \
 6  Ripon Grammar School (Community 11 -18)  117 (including 14 boarders)   
 
   PAN (sixth form)     Year  Number on roll  Total applications received  \
 6               30  2019-20           929.0                        477.0   
 
    Places allocated Last child criteria Distance of last child allocated  \
 6             117.0